# 🏀 NBA Foul Detection & Prediction — Complete End-to-End Interactive Pipeline
**Course:** CSC 434 — Artificial Intelligence & Machine Learning Final Project  
**Architecture:** Cross-Platform Dual-Stream Spatiotemporal `R(2+1)D-18` 3D CNN + `YOLOv11-Pose` 12-D Biomechanical Contact Encoder  

---
### Overview & How to Run Cell-by-Cell
This self-contained Jupyter Notebook consolidates the entire **4-stage NBA Foul Classifier pipeline** (`1_scrape_l2m.py`, `2_preprocess_tubes.py`, `3_model_and_dataset.py`, and `4_train_eval.py`) into one interactive, step-by-step workflow designed to run smoothly on **Windows, macOS (Intel & Apple Silicon M1/M2/M3/M4), and Linux (CUDA / CPU)**.

| Stage | Module Equivalent | Core Responsibility | Default Setting |
| :--- | :--- | :--- | :--- |
| **Cell 1** | Environment & Config | Creates `data/raw/{foul,no_foul}` & `data/processed/{foul,no_foul}`, selects device (`CUDA`/`MPS`/`CPU`) | `BATCH_SIZE = 2`, `ACCUM_STEPS = 2` |
| **Cell 2** | `1_scrape_l2m.py` | Live NBA L2M scraper & MP4 downloader into `data/raw/{foul,no_foul}/` (`RUN_LIVE_WEB_SCRAPE = True`) | `MAX_GAMES = 10`, `MAX_CLIPS = 30` |
| **Cell 3** | Exploratory Data Analysis | Inspects `l2m_metadata.csv`, `l2m_incorrect_fouls.csv`, and counts downloaded `.mp4` clips in `data/raw/` | Live dataset + MP4 audit |
| **Cell 4** | `2_preprocess_tubes.py` (Core) | `YOLOv11-Pose` 17-keypoint skeleton analysis, 12-D biomechanical contact features, & EMA ROI smoothing | `MAX_SCAN_SAMPLES = 12`, `YOLO_BATCH = 4` |
| **Cell 5** | `2_preprocess_tubes.py` (Runner) | Two-pass low-RAM action-tube extractor over downloaded `data/raw/` MP4s (with offline fallback) | `MAX_PREPROCESS_CLIPS = 24` |
| **Cell 6** | `3_model_and_dataset.py` | `L2MFoulDataset`, `ConsistentVideoAugmentor`, `BiomechanicalFoulEncoder`, & `FoulClassifier3D` (`R(2+1)D-18`) | 512-D 3D CNN + 64-D Pose Fusion |
| **Cell 7** | `4_train_eval.py` (Train/Eval) | `FoulFocalBCEWithLogitsLoss` (`gamma=1.5`), gradient accumulation, & dynamic F1 threshold calibration | `EPOCHS = 5`, `MAX_TRAIN_SAMPLES = 30` |
| **Cell 8** | `4_train_eval.py` (Inference) | Single-clip inference on a real `data/raw/` MP4 clip & annotated `.mp4` overlay export | Cross-OS codec fallback |

In [ ]:
# ==============================================================================
# CELL 1: Cross-Platform Environment Setup, Folder Creation & Configuration
# ==============================================================================
import os
import sys
import csv
import gc
import html
import json
import time
import random
import hashlib
import logging
import platform
import contextlib
from pathlib import Path
from collections import Counter
from typing import Any, Callable, Dict, List, Optional, Set, Tuple

# Prevent OpenMP / MKL / OpenCV FFmpeg thread deadlocks on Windows, macOS, and Linux
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.transforms.functional as TF
from torchvision.models.video import R2Plus1D_18_Weights, r2plus1d_18
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

cv2.setNumThreads(0)

try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None

# Resolve project directory automatically whether opened from repo root or NBA-Foul-Classifier-WIP/
CWD = Path.cwd().resolve()
if (CWD / "NBA-Foul-Classifier-WIP").exists():
    PROJECT_DIR = CWD / "NBA-Foul-Classifier-WIP"
else:
    PROJECT_DIR = CWD

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
METADATA_CSV = DATA_DIR / "l2m_metadata.csv"
INCORRECT_CSV = DATA_DIR / "l2m_incorrect_fouls.csv"
PROCESSED_METADATA_CSV = PROCESSED_DIR / "processed_metadata.csv"
CHECKPOINT_PATH = PROJECT_DIR / "best_foul_r2plus1d.pt"

# Immediately create data/raw/{foul,no_foul} and data/processed/{foul,no_foul}
for sub in ("foul", "no_foul"):
    (RAW_DIR / sub).mkdir(parents=True, exist_ok=True)
    (PROCESSED_DIR / sub).mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------------------
# CONFIGURATION KNOBS (Balanced for scraping multiple games & running on laptops)
# Increase max_games / max_scrape_clips / max_preprocess_clips for even more data!
# ------------------------------------------------------------------------------
CONFIG = {
    "max_games": 10,             # Inspect up to 10 NBA L2M game reports (Scale up to 25-100+)
    "max_scrape_clips": 30,      # Download up to 30 raw MP4 clips into data/raw/ (Scale up to 200+)
    "include_archive": True,     # Also search historical NBA L2M archive pages for more games
    "max_preprocess_clips": 24,  # Preprocess up to 24 clips (Set to 0 to process all clips)
    "max_live_play_sec": 15.0,   # Scan first 15s of broadcast clip (skips multi-minute replay reviews)
    "max_scan_samples": 12,      # 12 pose scan frames per clip in Pass 1 (Scale up to 24)
    "yolo_batch_size": 4,        # 4 frames per YOLOv11-Pose call (Scale up to 8-16 on GPU)
    "num_frames": 16,            # 16 frames sampled over 1.5s window around peak foul contact
    "crop_size": 224,            # 224x224 spatial action-tube crop
    "batch_size": 2,             # Micro-batch size per forward pass (Scale up to 4-8 on GPU)
    "accum_steps": 2,            # Gradient accumulation steps (Effective batch size = 2 * 2 = 4)
    "epochs": 5,                 # Default: 5 epochs (Scale up to 20-30 for full convergence)
    "max_train_samples": 30,     # Default: 30 samples (Set to 0 to train on full dataset)
    "lr": 1e-4,
    "weight_decay": 1e-2,
    "dropout": 0.4,
    "seed": 42,
}

def select_compute_device(requested: Optional[str] = None) -> torch.device:
    if requested and requested.lower() != "auto":
        return torch.device(requested)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def get_safe_num_workers(requested_workers: int = 0) -> int:
    if platform.system() in ("Darwin", "Windows"):
        return 0
    return max(0, requested_workers)

DEVICE = select_compute_device()
print(f"Operating System : {platform.system()} ({platform.machine()})")
print(f"PyTorch Version  : {torch.__version__}")
print(f"Selected Device  : {DEVICE}")
print(f"Raw Video Folder : {RAW_DIR} (exists={RAW_DIR.exists()})")
print(f"Processed Folder : {PROCESSED_DIR} (exists={PROCESSED_DIR.exists()})")

---
## Stage 1: NBA Last Two Minute (L2M) Scraper & Video Downloader (`1_scrape_l2m.py`)
Official NBA Last Two Minute (L2M) reports evaluate every officiating decision inside the final 2 minutes of Q4 and Overtime:
- **Positive Class (`1 = foul`)**: `CC` (*Correct Call*) and `INC` (*Incorrect Non-Call* — an uncalled foul that actually occurred).
- **Negative Class (`0 = no_foul`)**: `CNC` (*Correct Non-Call* — legal guarding contact) and `IC` (*Incorrect Call* — a phantom whistle on legal defense).

Running the cell below with `RUN_LIVE_WEB_SCRAPE = True` will:
1. Create `data/raw/foul/` and `data/raw/no_foul/`.
2. Scrape up to `CONFIG['max_games']` (`10`) NBA L2M reports (across current season and archive pages) while preserving all existing rows in `data/l2m_metadata.csv`.
3. Download up to `CONFIG['max_scrape_clips']` (`30`) `.mp4` broadcast video clips into `data/raw/{foul,no_foul}/` (interleaving `foul` and `no_foul` clips so both folders get populated!).

In [ ]:
# ==============================================================================
# CELL 2: Stage 1 — Live NBA L2M Web Scraper & Raw MP4 Video Downloader
# ==============================================================================
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
scrape_mod = importlib.import_module("1_scrape_l2m")
importlib.reload(scrape_mod)

RUN_LIVE_WEB_SCRAPE = True  # Enabled by default so data/raw/ is populated with MP4 clips!

if RUN_LIVE_WEB_SCRAPE:
    scraped_rows = scrape_mod.run_training_dataset_scraper(
        output_dir=RAW_DIR,
        metadata_csv=METADATA_CSV,
        daily_csv=INCORRECT_CSV,
        include_archive=CONFIG["include_archive"],
        max_games=CONFIG["max_games"],
        max_clips=CONFIG["max_scrape_clips"],
        skip_download=False,
    )

# Always keep l2m_incorrect_fouls.csv synchronized with all foul plays in l2m_metadata.csv
populated_foul_rows = scrape_mod.populate_foul_csv_from_local_metadata(
    metadata_csv=METADATA_CSV,
    daily_csv=INCORRECT_CSV,
    include_confirmed_fouls=True,
)

raw_foul_mp4s = sorted((RAW_DIR / "foul").glob("*.mp4"))
raw_nofoul_mp4s = sorted((RAW_DIR / "no_foul").glob("*.mp4"))
print(f"\nRaw MP4 Videos in {RAW_DIR}:")
print(f"  • data/raw/foul/    : {len(raw_foul_mp4s)} MP4 clips")
print(f"  • data/raw/no_foul/ : {len(raw_nofoul_mp4s)} MP4 clips")
print(f"  • Total Raw Videos  : {len(raw_foul_mp4s) + len(raw_nofoul_mp4s)} MP4 clips")

---
## Exploratory Data Analysis (EDA): Inspecting the L2M Foul Dataset & Downloaded Videos
Let's inspect the class balance, review decisions (`CNC`, `CC`, `INC`, `IC`), foul types (`Shooting Foul` vs. `Personal Foul`), and verify which `.mp4` clips are cached in `data/raw/`.

In [ ]:
# ==============================================================================
# CELL 3: Dataset Summary, Class Imbalance & Raw MP4 Inventory
# ==============================================================================
with open(METADATA_CSV, "r", encoding="utf-8") as f:
    meta_rows = list(csv.DictReader(f))

label_counts = Counter(int(r["label"]) for r in meta_rows)
decision_counts = Counter(r["review_decision"] for r in meta_rows)
call_counts = Counter(r["call_type"] for r in meta_rows)
unique_games = sorted(set(r["game_id"] for r in meta_rows))

print(f"Total Labeled Plays in {METADATA_CSV.name}: {len(meta_rows)} across {len(unique_games)} NBA games")
print(f"Class Distribution : no_foul (0) = {label_counts[0]} ({label_counts[0]/max(1, len(meta_rows))*100:.1f}%) | "
      f"foul (1) = {label_counts[1]} ({label_counts[1]/max(1, len(meta_rows))*100:.1f}%)")
print(f"Imbalance Ratio    : {label_counts[0] / max(1, label_counts[1]):.2f} : 1 (no_foul : foul)")
print(f"Review Decisions   : {dict(decision_counts)}")
print(f"Foul Call Types    : {dict(call_counts)}")

downloaded_mp4s = list(RAW_DIR.rglob("*.mp4"))
print(f"\nDownloaded MP4 files ready in data/raw/: {len(downloaded_mp4s)}")
for mp4_file in downloaded_mp4s[:6]:
    size_mb = mp4_file.stat().st_size / (1024 * 1024)
    print(f"  • {mp4_file.relative_to(PROJECT_DIR)} ({size_mb:.2f} MB)")

---
## Stage 2: Biomechanical Pose Estimation & Memory-Safe Action-Tube Extraction (`2_preprocess_tubes.py`)
### Our Two-Pass Cross-Platform Solution:
- **Pass 1 (Lightweight Pose Scan)**: Caps scanning to the first `15.0` seconds of live play, decodes only `12` downscaled `640x360` frames (< 8 MB RAM), closes `cv2.VideoCapture` cleanly, and runs `YOLOv11-Pose` (`yolo11n-pose.pt`) in micro-batches (`yolo_batch_size = 4`).
- **12-D Biomechanical Contact Vector**: Computes wrist/elbow proximity, defender arm intrusion into shooter upper body, torso collision proximity, bounding-box IoU, vertical velocity, defender closing velocity, shooter gather elevation, defender verticality, ball proximity, paint crowding, height offset, and composite contact intensity.
- **Pass 2 (Stabilized 16-Frame Crop)**: Smooths ROI boxes via bidirectional EMA and decodes **only the 16 frames** in a `1.5s` window around peak contact at `224x224` resolution.
- **Prioritizes Real Downloaded MP4s**: Automatically processes all real `.mp4` videos in `data/raw/` first (and overwrites any older synthetic `.pt` tensors with real `YOLOv11-Pose` tensors!).

In [ ]:
# ==============================================================================
# CELL 4: Inspect Stage 2 Biomechanical Pose & Action-Tube Functions
# ==============================================================================
preprocess_mod = importlib.import_module("2_preprocess_tubes")
importlib.reload(preprocess_mod)

FEATURE_NAMES_12D = [
    "0: Arm-to-Arm Proximity",
    "1: Defender Arm -> Shooter Upper-Body Intrusion",
    "2: Torso/Hip Collision Proximity",
    "3: Pair Bounding-Box IoU",
    "4: Vertical Elevation Velocity",
    "5: Defender Closing Velocity",
    "6: Shooter Gather Arm Elevation",
    "7: Defender Verticality Arm Elevation",
    "8: Ball-to-Pair Proximity",
    "9: Paint Crowding Density",
    "10: Vertical Height Offset",
    "11: Composite Foul-Contact Score",
]
print("12-D Biomechanical Foul Feature Schema:")
for fname in FEATURE_NAMES_12D:
    print(f"  • {fname}")

In [ ]:
# ==============================================================================
# CELL 5: Run Stage 2 Preprocessing on Downloaded Raw MP4 Videos
# ==============================================================================
# Prioritizes real .mp4 clips in data/raw/ and replaces any old synthetic .pt files
processed_rows = preprocess_mod.preprocess_dataset(
    metadata_csv=METADATA_CSV,
    processed_dir=PROCESSED_DIR,
    processed_metadata_csv=PROCESSED_METADATA_CSV,
    window_sec=1.5,
    num_frames=CONFIG["num_frames"],
    crop_size=CONFIG["crop_size"],
    device=str(DEVICE),
    max_live_play_sec=CONFIG["max_live_play_sec"],
    max_scan_samples=CONFIG["max_scan_samples"],
    max_clips=CONFIG["max_preprocess_clips"],
    yolo_batch_size=CONFIG["yolo_batch_size"],
    synthetic_fallback=True,
)
print(f"\nPreprocessed {len(processed_rows)} balanced clips -> {PROCESSED_METADATA_CSV}")

---
## Stage 3: Dual-Stream Spatiotemporal Architecture (`3_model_and_dataset.py`)
Our classifier fuses two complementary representations:
1. **Visual Spatiotemporal Stream (`R(2+1)D-18`)**: Processes the `(3, 16, 224, 224)` action tube into a `512-D` embedding (`stem`, `layer1`, `layer2` frozen; `layer3` and `layer4` fine-tuned).
2. **Biomechanical Pose Stream (`BiomechanicalFoulEncoder`)**: A two-layer `Conv1d` network (`12 -> 32 -> 64` channels) with **Temporal Contact-Frame Attention Pooling** over the `(16, 12)` pose feature sequence, producing a `64-D` kinematic embedding.
3. **Late Fusion Classification Head**: Concatenates `[512-D visual || 64-D pose] -> 576-D` through `LayerNorm -> Dropout(0.4) -> Linear(576, 128) -> GELU -> Dropout(0.2) -> Linear(128, 1)`.

In [ ]:
# ==============================================================================
# CELL 6: Verify Dataset Loading & Dual-Stream FoulClassifier3D Forward Pass
# ==============================================================================
model_mod = importlib.import_module("3_model_and_dataset")
importlib.reload(model_mod)

ConsistentVideoAugmentor = model_mod.ConsistentVideoAugmentor
L2MFoulDataset = model_mod.L2MFoulDataset
FoulClassifier3D = model_mod.FoulClassifier3D

demo_dataset = L2MFoulDataset(
    processed_dir=PROCESSED_DIR,
    metadata_csv=PROCESSED_METADATA_CSV,
    is_train=True,
    max_samples=CONFIG["max_train_samples"],
    return_foul_features=True,
)

sample_video, sample_feats, sample_label = demo_dataset[0]
print(f"Loaded Dataset Size  : {len(demo_dataset)} samples")
print(f"Sample Video Tensor  : shape={tuple(sample_video.shape)}, dtype={sample_video.dtype}")
print(f"Sample Pose Features : shape={tuple(sample_feats.shape)}, dtype={sample_feats.dtype}")
print(f"Sample Ground Truth  : label={int(sample_label.item())} ({'foul' if sample_label.item() == 1 else 'no_foul'})")

test_model = FoulClassifier3D(pretrained=False, dropout_p=CONFIG["dropout"], freeze_early=True).to(DEVICE)
param_summary = test_model.trainable_parameter_summary()

with torch.no_grad():
    sample_logit = test_model(
        sample_video.unsqueeze(0).to(DEVICE),
        sample_feats.unsqueeze(0).to(DEVICE),
    )
    sample_prob = torch.sigmoid(sample_logit).item()

print(f"\nModel Parameter Breakdown:")
print(f"  • Total Parameters    : {param_summary['total_params']:,}")
print(f"  • Trainable Parameters: {param_summary['trainable_params']:,} (layer3, layer4, pose_encoder, classifier)")
print(f"  • Frozen Parameters   : {param_summary['frozen_params']:,} (stem, layer1, layer2)")
print(f"Forward Pass Check      : logit={sample_logit.item():.4f} -> P(foul)={sample_prob*100:.2f}%")

---
## Stage 4: Lightweight Training Loop, Focal Loss & Threshold Calibration (`4_train_eval.py`)
We train with class-balanced `FoulFocalBCEWithLogitsLoss` (`gamma = 1.5`), micro-batch gradient accumulation (`batch_size = 2`, `accum_steps = 2`), and per-epoch validation F1 threshold calibration.

In [ ]:
# ==============================================================================
# CELL 7: Train & Calibrate the Dual-Stream Foul Classifier
# ==============================================================================
train_mod = importlib.import_module("4_train_eval")
importlib.reload(train_mod)

train_mod.seed_everything(CONFIG["seed"])

all_samples = demo_dataset.samples
labels = [lbl for _, lbl in all_samples]
stratify_labels = labels if len(set(labels)) > 1 and min(labels.count(0), labels.count(1)) >= 2 else None

train_samples, val_samples = train_test_split(
    all_samples,
    test_size=0.25,
    random_state=CONFIG["seed"],
    stratify=stratify_labels,
)

train_ds = L2MFoulDataset(samples=train_samples, is_train=True, return_foul_features=True)
val_ds = L2MFoulDataset(samples=val_samples, is_train=False, return_foul_features=True)

safe_workers = get_safe_num_workers(0)
train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=safe_workers)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=safe_workers)

USE_PRETRAINED_KINETICS = True  # Automatically falls back to random init if offline
model = FoulClassifier3D(
    pretrained=USE_PRETRAINED_KINETICS,
    dropout_p=CONFIG["dropout"],
    freeze_early=True,
).to(DEVICE)

pos_weight = train_mod.compute_pos_weight(train_samples, device=DEVICE)
criterion = train_mod.FoulFocalBCEWithLogitsLoss(pos_weight=pos_weight, gamma=1.5)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = CosineAnnealingLR(optimizer, T_max=max(1, CONFIG["epochs"]), eta_min=CONFIG["lr"] * 0.01)
scaler = torch.amp.GradScaler("cuda", enabled=True) if (DEVICE.type == "cuda" and torch.cuda.is_available()) else None

history = []
best_f1 = -1.0

for epoch in range(1, CONFIG["epochs"] + 1):
    t_loss = train_mod.train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE,
        accum_steps=CONFIG["accum_steps"],
    )
    v_loss, metrics = train_mod.evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=DEVICE,
    )
    scheduler.step()
    history.append({"epoch": epoch, "train_loss": t_loss, "val_loss": v_loss, **metrics})
    print(
        f"Epoch [{epoch:02d}/{CONFIG['epochs']:02d}] | Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | "
        f"Acc: {metrics['accuracy']:.3f} | F1: {metrics['f1']:.3f} | ROC-AUC: {metrics['roc_auc']:.3f} | "
        f"Optimal Thresh: {metrics['threshold']:.2f}"
    )
    if metrics["f1"] >= best_f1:
        best_f1 = metrics["f1"]
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "metrics": metrics,
                "optimal_threshold": metrics["threshold"],
            },
            CHECKPOINT_PATH,
        )

print(f"\nSaved calibrated checkpoint -> {CHECKPOINT_PATH}")

---
## Stage 5: Single-Clip Foul Prediction & Annotated MP4 Video Overlay
The cell below evaluates the trained checkpoint on validation data and automatically runs `predict_single_clip()` on a downloaded `.mp4` broadcast video in `data/raw/` to generate an annotated `*_annotated.mp4` video overlay!

In [ ]:
# ==============================================================================
# CELL 8: Inference & Annotated MP4 Video Overlay Generation
# ==============================================================================
ckpt = model_mod.load_torch_file_safe(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"], strict=False)
calibrated_threshold = float(ckpt.get("optimal_threshold", 0.50))
model.eval()

val_path, val_true_label = val_samples[0]
val_video, val_feats, _ = val_ds[0]

with torch.no_grad():
    logit = model(val_video.unsqueeze(0).to(DEVICE), val_feats.unsqueeze(0).to(DEVICE))
    foul_prob = float(torch.sigmoid(logit).item())

verdict = "FOUL DETECTED" if foul_prob >= calibrated_threshold else "LEGAL CONTACT / NO FOUL"
peak_frame_idx = int(torch.argmax(val_feats[:, 11]).item())
peak_contact_score = float(val_feats[peak_frame_idx, 11].item())

print(f"Evaluated Clip       : {val_path.name}")
print(f"Ground-Truth Label   : {val_true_label} ({'FOUL' if val_true_label == 1 else 'NO FOUL'})")
print(f"Predicted Probability: {foul_prob * 100:.2f}% (Calibrated Threshold: {calibrated_threshold * 100:.0f}%)")
print(f"Model Verdict        : {verdict}")
print(f"Peak Contact Frame   : Frame {peak_frame_idx + 1}/16 (Biomechanical Score = {peak_contact_score * 100:.1f}%)")

# Run full MP4 video overlay inference on a raw MP4 clip in data/raw/
existing_mp4s = [p for p in sorted(RAW_DIR.rglob("*.mp4")) if "_annotated" not in p.name]
if existing_mp4s and YOLO is not None:
    sample_mp4 = existing_mp4s[0]
    print(f"\nRunning full MP4 video overlay inference on {sample_mp4.relative_to(PROJECT_DIR)}...")
    train_mod.predict_single_clip(video_path=sample_mp4, checkpoint_path=CHECKPOINT_PATH)
else:
    print("\n[Tip] To generate an annotated MP4 overlay on a raw broadcast video, run:")
    print("      python 4_train_eval.py --predict data/raw/foul/<play_id>.mp4")